In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# E.1 Five Factorizations, One Idea

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Epilogue",
    number="E.1",
    title="Five Factorizations, One Idea",
    blurb="The Prologue ended on a question about rank that no factorization "
    "could settle; forty-six notebooks later, the answer takes four lines "
    "and a theorem. The course closes by answering it, choosing methods "
    "for eight problems by procedure instead of habit, running one "
    "capstone five ways, and pricing every algorithm it taught on a "
    "single axis.",
    difficulty="advanced",
    estimate="90–120 min",
)

## Notebook overview

The [Prologue](../prologue/one-matrix-five-factorizations.ipynb) took
one $4\times4$ matrix apart five ways and left one question standing:
after a $10^{-12}$ perturbation, is the rank three or four? Every
answer was defensible then. Now it is a *derivation*: the perturbation
has a known operator norm, **Weyl's inequality** (proved by Volume
IV's machinery) says no singular value can move farther than that
norm, so any singular value *below* $3\lVert E\rVert_2$ is
indistinguishable from noise — and at that derived tolerance the rank
is **three**, gated, with the third singular value twelve orders above
the line and the fourth an order below it. The staircase from the
Prologue's final figure returns with the derived tolerance drawn on
it: the step the course now knows how to choose.

The rest is consolidation, all of it gated. A **decision procedure**
— five questions about a problem, in order — is encoded as a function
and applied to eight named problems from the course's volumes; its
choices match the canonical answers eight for eight. The
**Läuchli ladder** shows the accuracy hierarchy conditioning theory
predicts: normal equations track $\kappa^2\varepsilon$ through three
rungs (losing everything at $\kappa = 10^{8}$) while QR and SVD sit
at machine precision throughout. The **capstone** — a structured,
numerically rank-deficient least-squares problem — is solved five
ways, and the verdict is [§5.1](../05-numerical/norms-conditioning-stability.ipynb)'s
deepest lesson reprised: every route's *predictions* agree at the
noise level (residuals are blind), while solution norms differ by
$10^{5}$ and noise-sensitivity by $10^{7}$ — the methods differ not
in fit but in what they silently add from the null space. And the
**cost table**: measured times for matvec, solve, QR and SVD across
a size sweep, with the FLOP-model exponents gated as arithmetic and
the *fitted* timing exponents reported inside wide bands — the
manifest asked for 0.2-tight timing gates, and [§5.2](../05-numerical/eigenvalue-algorithms.ipynb)'s
own measurement of a "cubic" 1.71 is why the course declines, one
final time, to gate a clock.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Everything here was built earlier; the only new content
> is the assembly. Strang {cite}`strang2023`, Trefethen and Bau
> {cite}`trefethen1997`, Golub and Van Loan {cite}`golub2013` — the
> course's three companions, one last citation each.

## Theory in brief

### The one idea

Every factorization the course taught is the same move: **rewrite the
matrix as structured factors, then answer questions at the structure's
price.**

```{math}
:label: eq-ep-five
A = CR = LU = QR = Q\Lambda Q^{\top} = U\Sigma V^{\top},
```

rank and column space from $CR$; solves from $LU$; least squares and
orthonormal bases from $QR$; symmetric dynamics and quadratic forms
from the spectral theorem; and everything metric — norms, angles,
rank decisions, low-rank truth — from the SVD. The course's second
idea rode alongside: *what the mathematics determines, gate; what the
machine chooses, report* — and this notebook's gates are its final
demonstration.

### The tool that answers the Prologue

Weyl's perturbation inequality for singular values:

```{math}
:label: eq-ep-weyl
\bigl|\sigma_i(A + E) - \sigma_i(A)\bigr| \;\le\; \lVert E\rVert_2
\qquad \text{for every } i,
```

so a singular value below the perturbation's norm certifies nothing —
it may be exactly zero wearing noise — while one far above it is
real. A rank tolerance is therefore not a convention but an *estimate
of* $\lVert E\rVert_2$, and choosing it is a statement about the
data's noise, exactly as the Prologue said — now with the theorem
that makes the statement precise.

---
## Setup

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import legendre

from ecp import validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps

# The Prologue's matrix and its perturbation, rebuilt exactly.
A_PRO = np.array([[2.0, 1.0, 3.0, 0.0],
                  [3.0, 4.0, 7.0, 0.0],
                  [1.0, 2.0, 3.0, -1.0],
                  [0.0, 1.0, 1.0, 2.0]])
G_PRO = np.random.default_rng(0).standard_normal((4, 4))
E_PRO = 1e-12 * G_PRO
A_TILDE = A_PRO + E_PRO

## Exercise 1 — The Prologue's question, answered in four lines

**Part a)** Rebuild $\tilde A = A + 10^{-12}G$ exactly as the
Prologue did, and gate {eq}`eq-ep-weyl` on it: every singular value of
$\tilde A$ sits within $\lVert E\rVert_2 = 3.0\times10^{-12}$ of the
corresponding singular value of $A$ — the theorem, checked on the
matrix that motivated it.

**Part b)** Derive the tolerance the Prologue could not: the noise is
*known* here ($\lVert E\rVert_2$), so any singular value below
$\tau = 3\lVert E\rVert_2 \approx 9\times10^{-12}$ is
indistinguishable from a perturbed zero. Gate the resolution:
$\sigma_3 = 1.16 \gg \tau > \sigma_4 = 4.7\times10^{-13}$, and
`matrix_rank` at $\tau$ returns **3** — the answer, derived rather
than chosen.

**Part c)** Re-draw the Prologue's staircase with $\tau$ marked: the
figure that ended the course's first notebook, now with the step the
course's last notebook knows how to select.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    weyl_worst <= norm_E * (1 + 1e-10),
    "Weyl's inequality holds on the Prologue's perturbation (Eq. 2)",
    f"every singular value moved at most {weyl_worst:.1e} against the "
    f"bound {norm_E:.1e} — the theorem that turns noise into tolerance",
)
validate.check(
    rank_at_tau == 3 and sig_T[2] > 1e10 * TAU and sig_T[3] < TAU,
    "and at the derived tolerance the Prologue's rank is three, closed",
    f"sigma_3 sits {sig_T[2]/TAU:.0e}x above tau and sigma_4 "
    f"{TAU/sig_T[3]:.0f}x below — the answer is a derivation now, not "
    "a choice",
)

## Exercise 2 — The decision procedure, applied eight times

**Part a)** Encode the course's method-selection procedure as a
function of five problem attributes — square/rectangular?
symmetric? positive definite? sparse/structured? rank or spectrum
wanted? — returning one of the course's methods.

**Part b)** Apply it to eight named problems and gate the choices
against the canonical answers, eight for eight:

1. dense square nonsymmetric solve → **LU** ([§1.2](../01-matrices/elimination-lu.ipynb))
2. SPD solve → **Cholesky** ([§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb))
3. overdetermined least squares → **QR** ([§2.3](../02-orthogonality/least-squares-four-ways.ipynb))
4. rank-deficient least squares → **SVD** ([§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb))
5. few eigenpairs of a large sparse symmetric matrix →
   **Lanczos** ([§5.5](../05-numerical/krylov-gmres-preconditioning.ipynb))
6. huge sparse SPD solve → **CG** ([§5.4](../05-numerical/stationary-and-cg.ipynb))
7. huge sparse nonsymmetric solve → **GMRES** ([§5.5](../05-numerical/krylov-gmres-preconditioning.ipynb))
8. best rank-$k$ approximation → **truncated SVD**
   ([§4.2](../04-svd/low-rank-eckart-young.ipynb))

The gate is modest by design — a lookup agreeing with a list — but it
is the notebook's real deliverable: the procedure *is* the course,
compressed to one function.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    all_match,
    "the decision procedure matches the canonical choice, eight for "
    "eight",
    "square/symmetric/definite/sparse/goal in, method out — the course "
    "compressed to one function, and the function audited against its "
    "source",
)

## Exercise 3 — The accuracy hierarchy, on the Läuchli ladder

Conditioning theory ranks the least-squares routes:
normal equations square $\kappa$, orthogonal methods do not. The
Läuchli matrix — a row of ones over $\varepsilon I$ — makes the
ranking measurable at three rungs.

**Part a)** For $\varepsilon = 10^{-4}, 10^{-6}, 10^{-8}$
($\kappa \approx 2.4\times10^{4}, 10^{6}, 10^{8}$), solve the
consistent system by normal equations, Householder QR, and SVD;
record coefficient errors against the known solution.

**Part b)** Gate the hierarchy: QR and SVD below $10^{-14}$ at every
rung (backward stability pays no $\kappa^2$), while the
normal-equations error lands within a factor 100 of
$\kappa^2\varepsilon_{\text{mach}}$ at every rung — $10^{-8}$, then
$10^{-4}$, then **0.44**: at the last rung the Gram matrix has
numerically lost the information QR still holds, and the answer is
garbage from a routine that raised no error. The ordering the theory
predicts, measured across four orders of failure.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    orth_ok,
    "QR and SVD sit at machine precision on every rung",
    "backward stability pays kappa once, and a consistent system pays "
    "nothing more — 2.3's four-ways verdict, at its extreme",
)
validate.check(
    ne_tracks,
    "while normal equations track kappa-squared-epsilon through four "
    "orders of failure",
    f"errors {[f'{v['ne']:.0e}' for v in lauchli.values()]} within "
    "100x of kappa^2 eps at each rung — ending in 44% error from a "
    "routine that raised no warning: the course's case against the "
    "Gram matrix, closed",
)

## Exercise 4 — The capstone, five ways

One structured, numerically rank-deficient least-squares problem —
scaled Legendre features, 300 samples, 40 columns whose trailing ten
decay to $10^{-13}$, noise $10^{-6}$ — solved by:
**truncated SVD** at the derived tolerance, **`lstsq`** with matching
`rcond`, **ridge**, **CG** on regularized normal equations, and
unpivoted **QR** (the cautionary entry).

**Part a)** Gate the [§5.1](../05-numerical/norms-conditioning-stability.ipynb)
reprise: all five routes' *training predictions* agree with the clean
signal at the noise level (relative error below $10^{-5}$, within
$3\times$ of each other) — residuals cannot tell the methods apart,
which is the whole danger.

**Part b)** Gate what *does* tell them apart: the solution norms —
regularized routes $O(1)$, QR five orders larger (junk coefficients
in the numerically dead directions, one-sidedly gated as exceeding
$10^{3}\times$) — and the **noise sensitivity**: refit under a second
noise draw and compare; the truncated-SVD solution moves $O(1)$
while QR's moves by $10^{7}$ (ratio gated above $10^{3}$). Print the
full accuracy/norm/sensitivity/cost table: the course's habits, in
one grid.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    pred_close,
    "all five routes predict identically at the noise level — "
    "residuals are blind (5.1, reprised as the finale)",
    f"prediction errors within [{min(preds):.1e}, {max(preds):.1e}]: "
    "the fit cannot distinguish a careful method from a careless one, "
    "which is why the next two gates exist",
)
validate.check(
    norm_ratio > 1e3 and sens_ratio > 1e3,
    "while solution norms and noise sensitivity separate them by "
    "orders of magnitude",
    f"QR carries {norm_ratio:.0e}x the coefficient norm and "
    f"{sens_ratio:.0e}x the noise sensitivity of truncated SVD — what "
    "the methods add from the numerically dead subspace is the entire "
    "difference, and only the spectrum sees it",
)

## Exercise 5 — The price list, and the course's last rule

**Part a)** The FLOP models for the course's operations — matvec
$2n^2$, LU solve $\tfrac23n^3$, QR $2mn^2 - \tfrac23n^3$, SVD
$14mn^2 + 8n^3$ — are arithmetic and gated as such: their exponents
in $n$ (square case) are exactly $2, 3, 3, 3$.

**Part b)** Measure wall times across $n = 512 \dots 4096$ and *fit*
exponents. The manifest asked to gate them within 0.2 of the models;
[§5.2](../05-numerical/eigenvalue-algorithms.ipynb) measured a "cubic"
at 1.71 in exactly this size range and [§6.4](../06-structure/kronecker-vec-separable.ipynb)
watched one machine straddle a timing gate between two runs — so the
course declines, one final time, to gate a clock: fitted exponents
are **reported**, gated only inside wide sanity bands (matvec in
$[0.5, 2.2]$, the cubics in $[2.0, 3.6]$), and the *ordering*
(every cubic steeper than the matvec by at least 0.5) is gated
because BLAS constants cannot invert an exponent across a full
sweep. Draw the sweep — one axis, every solver the course taught.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


```{admonition} With your assistant
:class: tip
The decision procedure of Exercise 2 is eight rows deep; a course is
not a career. Ask your assistant to stress it: invent five problems
that FALL BETWEEN the rows (sparse and rank-deficient; symmetric but
indefinite; structured Toeplitz least squares; a spectrum wanted to
machine precision; a matrix too large to hold), ask the procedure for
its answer, and then check each answer against the course's own
notebooks rather than accepting it: (i) does the recommended method's
home notebook list the problem's structure among its assumptions?
(ii) does a gate from that notebook transfer to the new problem and
pass? (iii) where the procedure has no good row, say so — the honest
output of a decision procedure is sometimes 'this course has not
earned an answer', and finding those edges is the check. The check —
as it has been forty-six times before — is yours.
```

### Validation 5

In [ ]:
validate.check(
    all(model_exponents[k] == (2 if k == "matvec" else 3)
        for k in model_exponents),
    "the FLOP models' exponents are 2, 3, 3, 3 — arithmetic, gated",
    "the price list's structure is mathematics; only its constants "
    "belong to the machine",
)
validate.check(
    bands_ok and ordering_ok,
    "and the measured clocks respect the ordering, inside wide bands "
    "only",
    f"fitted exponents {({k: round(v, 2) for k, v in fitted.items()})} "
    "— reported per Rule 2, with the manifest's 0.2-tight timing gate "
    "declined for the same reason 5.2 measured a cubic at 1.71: the "
    "course's last gate is about what gates may claim",
)

---
## Notebook summary

**The Prologue's question is closed.** Weyl's inequality held on the
perturbed matrix (every singular value within
$3\times10^{-12}$ of its original), the tolerance
$\tau = 3\lVert E\rVert_2$ was *derived* from the known noise, and at
$\tau$ the rank is three — with $\sigma_3$ twelve orders above the
line and $\sigma_4$ an order below. The staircase's step is chosen,
and choosing it took a theorem, not a convention.

**The course compresses to a procedure that survives audit.** Eight
problems, eight canonical methods, eight matches — LU to Cholesky to
QR to SVD to Lanczos to CG to GMRES to truncated SVD — with the five
shapes drawn one last time above them.

**The hierarchy and the blindness both measured true.** Läuchli's
ladder had normal equations tracking $\kappa^2\varepsilon$ into 44%
error while QR and SVD held machine precision; the capstone's five
routes fit identically at the noise level while their solution norms
and noise sensitivities differed by $10^{5}$ and $10^{7}$ —
residuals are blind, spectra are not, and that pair of sentences is
Volume V compressed.

**And the last gate was about gating.** The price list's exponents
(2, 3, 3, 3) were gated as arithmetic; the measured clocks were
reported inside wide bands with their ordering gated — because the
course's deepest habit is not a factorization but a question: *is
this number determined by the mathematics, or by the machine?* Five
factorizations, one idea — and one rule for telling the truth about
both.

## Outlook

- **The course ends; the ledger stays open.** Every notebook's gates
  run in CI on hardware the author never touched — the honest form
  of "the code works", and the habit most worth keeping.
- **Where to go deeper.** Trefethen and Bau {cite}`trefethen1997`
  for the numerical spine; Golub and Van Loan {cite}`golub2013` as
  the reference shelf; Strang {cite}`strang2023` for the view from
  the four subspaces, where this course began.
- **What was not covered.** Randomized algorithms beyond
  [§4.4](../04-svd/randomized-svd-sketching.ipynb), tensor methods
  beyond Volume VII's chain, optimization beyond quadratics,
  and everything genuinely nonlinear — each one a course, and each
  one reachable from here.
- **The one idea, one last time.** Factor first; then ask. And gate
  what the mathematics determines, report what the machine chooses —
  on every matrix you ever touch.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()